In [ ]:
import os
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq

load_dotenv()

# 1. Define the agent state
class FarmlyState(TypedDict):
    messages: Annotated[list, add_messages]


# 2. Initialize Groq LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.3
)


# 3. Create the agent node
def farmly_node(state: FarmlyState):
    system_prompt = """
    You are Farmly, a friendly agricultural assistant for Indian farmers.

    Instructions:
    - Give simple, practical farming advice.
    - Respond in the language used by the farmer.
    - Support Telugu, Hindi and English.
    - Give clear, step-by-step answers.
    - Never invent weather forecasts, mandi prices, soil readings
      or government scheme eligibility.
    - If reliable information is unavailable, say so.
    """

    response = llm.invoke(
        [{"role": "system", "content": system_prompt}]
        + state["messages"]
    )

    return {"messages": [response]}


# 4. Build the LangGraph
builder = StateGraph(FarmlyState)

builder.add_node("farmly", farmly_node)

builder.add_edge(START, "farmly")
builder.add_edge("farmly", END)

graph = builder.compile()


# 5. Function to call the agent
def ask_farmly(question: str):
    result = graph.invoke({
        "messages": [HumanMessage(content=question)]
    })

    return result["messages"][-1].content

In [ ]:
import os
import tempfile
import base64

from fastapi import APIRouter, UploadFile, File, Form, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from dotenv import load_dotenv
from sarvamai import SarvamAI

load_dotenv()

router = APIRouter(prefix="/voice", tags=["Voice"])

client = SarvamAI(
    api_subscription_key=os.getenv("SARVAM_API_KEY")
)

LANGUAGES = {
    "Telugu": "te-IN",
    "Hindi": "hi-IN",
    "English": "en-IN"
}

SPEAKERS = {
    "Telugu": "kavitha",
    "Hindi": "shubh",
    "English": "shubh"
}


# -------------------------------
# 1. SPEECH TO TEXT
# -------------------------------

@router.post("/transcribe")
async def transcribe_audio(
    audio: UploadFile = File(...),
    language: str = Form("Telugu")
):
    if language not in LANGUAGES:
        raise HTTPException(400, "Unsupported language")

    audio_bytes = await audio.read()

    if not audio_bytes:
        raise HTTPException(400, "Empty audio file")

    if len(audio_bytes) > 10 * 1024 * 1024:
        raise HTTPException(413, "Audio file is too large")

    suffix = ".webm"
    if audio.filename:
        suffix = os.path.splitext(audio.filename)[1] or ".webm"

    try:
        with tempfile.NamedTemporaryFile(
            suffix=suffix
        ) as temp_file:

            temp_file.write(audio_bytes)
            temp_file.flush()

            with open(temp_file.name, "rb") as audio_file:
                result = client.speech_to_text.transcribe(
                    file=audio_file,
                    model="saaras:v4",
                    language_code=LANGUAGES[language],
                    mode="codemix"
                )

        return {
            "transcript": result.transcript,
            "language": language
        }

    except Exception as e:
        raise HTTPException(500, f"Speech recognition failed: {str(e)}")


# -------------------------------
# 2. TEXT TO SPEECH
# -------------------------------

class SpeakRequest(BaseModel):
    text: str
    language: str = "Telugu"


@router.post("/speak")
def speak_text(request: SpeakRequest):

    if request.language not in LANGUAGES:
        raise HTTPException(400, "Unsupported language")

    if not request.text.strip():
        raise HTTPException(400, "Text cannot be empty")

    if len(request.text) > 2500:
        raise HTTPException(400, "Text is too long")

    try:
        response = client.text_to_speech.convert_stream(
            text=request.text,
            model="bulbul:v4-flash",
            language_code=LANGUAGES[request.language],
            speaker=SPEAKERS[request.language],
            output_audio_codec="mp3"
        )

        def audio_generator():
            for chunk in response:
                audio_data = getattr(chunk, "audio", chunk)

                if isinstance(audio_data, str):
                    yield base64.b64decode(audio_data)
                elif isinstance(audio_data, bytes):
                    yield audio_data

        return StreamingResponse(
            audio_generator(),
            media_type="audio/mpeg"
        )

    except Exception as e:
        raise HTTPException(500, f"Speech generation failed: {str(e)}")

In [28]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

from voice import router as voice_router
from farmly_agent import ask_farmly

app = FastAPI(title="Farmly AI Backend")

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://127.0.0.1:5500",
        "http://localhost:5500",
        "http://localhost:5173"
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)

app.include_router(voice_router)


class ChatRequest(BaseModel):
    message: str
    language: str = "Telugu"


@app.get("/")
def home():
    return {"message": "Farmly AI backend is running!"}


@app.post("/chat")
def chat(request: ChatRequest):
    answer = ask_farmly(request.message)

    return {
        "answer": answer,
        "language": request.language
    }

ModuleNotFoundError: No module named 'voice'